---
title: Lab 3. A quarterly forecast round
course: Example course (pymetrix)
links: [forecast-workflow, var, bayesian-var, reports]
---

# Lab 3. A quarterly forecast round

A forecasting team's quarter, from start to finish: forecast with a VAR, add judgement and record why, compare this round with the last one, check the model's track record against a Bayesian alternative, write the report, and save it all as a project that rebuilds itself.

The data are the simulated economy of Lab 2, in logs: output `ly`, consumption `lc` and investment `li`. The data end in 2024Q4, so this is the December round; the June round was made with data to 2024Q2.

In [ ]:
import numpy as np

from pymetrix import VAR, Bayes, Dataset, ForecastRound, ForecastStore, Minnesota, Report, backtest

data = Dataset.example("sample_macro").with_series(ly="log(y)", lc="log(c)", li="log(i)")
data

## 1. Two rounds

The June round used the data to 2024Q2; the December round uses everything. Each round records when it was made, the last period of data it used, and its source.

In [ ]:
june = data.select(span=(None, "2024Q2"))
june_round = ForecastRound.from_forecast("2024Q2 round", VAR(["ly", "lc", "li"], lags=2).fit(june).forecast(june, ("2024Q3", "2026Q4")),
                                         data=june)

var = VAR(["ly", "lc", "li"], lags=2).fit(data)
december = ForecastRound.from_forecast("2024Q4 round", var.forecast(data, ("2025Q1", "2026Q4")), data=data,
                                       notes="Model forecast before judgement.")
december.table(["ly"])

## 2. Judgement

A strike in the ports will cut output in early 2025, which the model cannot know. The adjustment is logged with its reason and the path before it, so the published forecast can always be separated from the model's.

In [ ]:
december = december.adjust("ly", {"2025Q1": -0.004, "2025Q2": -0.002}, reason="a strike in the ports in early 2025")
december.judgement()

In [ ]:
december.plot(data, ["ly", "lc"])

## 3. What changed since June?

A store is a folder of rounds. Comparing two rounds shows the revisions; the store's accuracy table scores past rounds against the data that came in since.

In [ ]:
from pathlib import Path
import tempfile

folder = Path(tempfile.mkdtemp())
store = ForecastStore(folder / "forecasts")
store.save(june_round)
store.save(december)
revisions = store.compare("2024Q2 round", "2024Q4 round", ["ly"])
revisions

In [ ]:
store.accuracy("ly", data)

**Question.** The December round is higher than the June round all the way out. Is that the new data, the judgement, or both? (Compare with `december.base`, the paths before judgement.)

## 4. Would a Bayesian VAR have done better?

A backtest re-estimates each model at every forecast origin from 2015 to 2022 and forecasts four quarters ahead. The Diebold-Mariano test (with the Harvey-Leybourne-Newbold correction) asks whether the difference in accuracy is more than chance.

In [ ]:
bvar = VAR(["ly", "lc", "li"], lags=2).fit(data, method=Bayes(prior=Minnesota(tightness=0.2)))
bt_var = backtest(var, data, ("2015Q1", "2022Q4"), horizon=4)
bt_bvar = backtest(bvar, data, ("2015Q1", "2022Q4"), horizon=4)
bt_var.accuracy(["ly"])

In [ ]:
bt_bvar.accuracy(["ly"])

In [ ]:
dm = bt_var.compare(bt_bvar, "ly", horizon=4)
dm

**Question.** Which model forecasts output better four quarters ahead, and is the difference significant? Try `tightness=0.05` and `1.0`: what does more shrinkage do?

## 5. A structural scenario

A VAR forecasts; a structural model answers "what if". Lab 2's estimated equations and the national-accounts identity make a small model. Solved past the end of the data, with government spending held at its last value, it gives a baseline; raising spending by 5 gives a fiscal scenario. Each becomes a round in the store, so the scenario reads as a revision to the baseline.

In [ ]:
from pymetrix import Equation, Model, OLS, Scenario

model = Model([Equation("c ~ y + c[-1]").fit(data, method=OLS(cov="hac")), Equation("i ~ y[-1]").fit(data), Equation("y = c + i + g")])
g_now = float(data["g"].iloc[-1])
horizon = ("2025Q1", "2026Q4")
baseline = model.solve(data, horizon, scenario=Scenario(set={"g": g_now}))
fiscal = model.solve(data, horizon, scenario=Scenario(set={"g": g_now + 5}))
for name, solution in (("2024Q4 model baseline", baseline), ("2024Q4 fiscal scenario", fiscal)):
    store.save(ForecastRound.from_forecast(name, solution.frame.loc["2025Q1":], variables=["y", "c", "i"], data=data,
                                           notes="Structural model; g held at its last value" + (" plus 5" if "fiscal" in name else "")))
scenario = store.compare("2024Q4 model baseline", "2024Q4 fiscal scenario", ["y"])
scenario

**Question.** The revision column is the effect on output of 5 more in government spending. Why does it grow over the first year, and how does it compare with Lab 2's multiplier?

## 6. The report

The report collects the fan chart, the judgement, the revisions, the track record and the fiscal scenario, and the model, and ends with an appendix of where the model's numbers came from: data, sample, method and pymetrix version. Save it as `.docx` too if python-docx is installed.

In [ ]:
report = Report("Quarterly forecast, 2024Q4", author="Lab 3")
report.heading("The forecast").text("Output, consumption and investment, with the judgement applied in 2025.")
report.add(december.plot(data, ["ly"]), caption="Output: fan chart")
report.add(december.judgement())
report.add(revisions)
report.heading("The model").add(var, caption="VAR in ly, lc, li")
report.heading("Track record").add(bt_var.accuracy(["ly"]))
report.add(dm)
report.heading("A fiscal scenario").add(scenario)
saved = [report.save(folder / f"forecast.{ext}") for ext in ("html", "xlsx")]
[p.name for p in saved]

The same round as a standard layout: the `forecast_round` template takes each result's fan chart,
paths and judgement, here in the `policy` theme, and Markdown is one of the formats.

In [ ]:
standard = Report.from_template("forecast_round", {"December round": december}, title="Quarterly forecast, 2024Q4",
                                theme="policy", span=("2022Q1", None))
standard.save(folder / "forecast_round.md").name

## 7. As a project

The same steps as the **Forecast** menu runs them. Each is recorded as Python; a project saves the data and the steps, and opening it replays them.

In [ ]:
from pymetrix.commands import AdjustRound, FitVAR, Forecast, MakeRound, Session, load_project, save_project

s = Session()
s.objects["data"] = data
s.run(FitVAR("ly, lc, li", lags="2"))
s.run(Forecast("var", "2025Q1", "2026Q4"))
s.run(MakeRound("forecast", "2024Q4 round"))
s.run(AdjustRound("forecast_round", "ly", "2025Q1=-0.004, 2025Q2=-0.002", "a strike in the ports in early 2025"))
print(s.script())

In [ ]:
path = save_project(s, folder / "lab3.pymx")
reopened = load_project(path)
same = np.allclose(reopened["forecast_round"].path("ly"), s["forecast_round"].path("ly"))
print(f"saved to {path.name}; reopened round identical: {same}")

## 8. Nowcasting the quarter in progress

The VAR forecast 2025Q1 from quarterly data. By mid-February, monthly indicators of the same
economy already cover January: a survey, employment and retail sales, with industrial production
a month behind (each series knows how long its figures take to come out). A bridge equation
averages the months the quarter has, after forecasting the missing ones; a dynamic factor model
uses every figure as it is, and shows which new figures moved its nowcast.

In [ ]:
from pymetrix import Bridge, DynamicFactor

quarterly = data.with_series(gy="400*dlog(y)")
monthly = Dataset.example("sample_monthly").with_series(gr="1200*dlog(retail)", gip="1200*dlog(ip)", gemp="1200*dlog(emp)")
known = monthly.as_of("2025-02-15")
known.ragged_edge("2025-02-15")

In [ ]:
bridge = Bridge("gy ~ gr + pmi").fit(quarterly, monthly)
february = DynamicFactor(monthly=["gr", "gip", "gemp", "pmi"], quarterly=["gy"]).fit(quarterly, known)
april = february.update(quarterly, monthly.as_of("2025-04-15"))  # the whole quarter's figures, same model
news = april.news(february, "gy")
assert abs(news.after - news.before - news.details["impact"].sum() - news.revisions) < 1e-9  # the news adds up
{"bridge, 15 Feb": round(float(bridge.nowcast(quarterly, known).iloc[0]), 2),
 "factor model, 15 Feb": round(news.before, 2), "factor model, 15 Apr": round(news.after, 2),
 "VAR (annualised growth)": round(400 * float(december.paths["ly"].loc["2025Q1"] - data["ly"].iloc[-1]), 2)}

In [ ]:
news.by_series.round(3)  # which series moved the nowcast between February and April

**Question.** The nowcasts start well above the VAR and come down as the quarter's figures arrive.
Which series did most of that, and would you change the December round's judgement for 2025Q1?